# Imports

In [5]:
import os
import dask.distributed 
from dask.distributed import LocalCluster, Client
from dask_jobqueue import SLURMCluster #trouble getting this to work
from pathml.core import HESlide
from pathml.ml import TileDataset
from pathml.preprocessing import Pipeline
from pathml.preprocessing import TissueDetectionHE, LabelWhiteSpaceHE, LabelArtifactTileHE, StainNormalizationHE
from torch.utils.data import DataLoader
import sys
import cv2   #For blur detection
from pathlib import Path
from PIL import Image
import numpy as np
import time
scratch = os.getenv('SLURM_SCRATCH')
sp = Path(scratch)
data = Path('/ix/rbao/Projects/panCancer_HE/data/skin/data/HE/')
results = Path('/ix/rbao/Projects/panCancer_HE/results/skin_rbao/h5')
data.exists()
dask.config.set({'temporary_directory': str(sp.joinpath('dask-worker-space'))}) #Crucial for reducing timeout issues

In [2]:
%%bash
cp "/ix/rbao/Projects/panCancer_HE/data/skin/data/HE/MEL_SYS10-577694_H&E.svs" $SLURM_SCRATCH'/MEL_SYS10-577694_H&E.svs'
echo "Finished"

Finished


# Load test image

In [2]:
fn = sp.joinpath('MEL_SYS10-577694_H&E.svs')
wsi = HESlide(fn)

In [3]:
blank_detect=LabelWhiteSpaceHE(label_name='ignore', proportion_threshold=0.9) #Thresh too low?
art_detect = LabelArtifactTileHE(label_name='ignore')
tissue_detect = TissueDetectionHE(mask_name='tissue',outer_contours_only= True, blur_ksize=21,threshold = 20) #Thresh possibly off for smaller tile size (<=
normalize = StainNormalizationHE(target = "normalize",
                                 stain_estimation_method = "macenko")
#TODO: implement blur detection as a pathml-compatible preprocessing step
#TODO: add tile coordinates or other 

In [ ]:
Pipeline(

# TODO Create blur transform

In [ ]:
class BlurDetect(Transform):
    """Detect blurry tiles."""
    def __init__(self,blur_thresh = 40, blur_label='ignore'):
        self.blur_thresh = blur_thresh
        self.blur_label = blur_label
        self.bgr = (2,1,0)
    def F(self, image):
        return cv2.Laplacian(image[:,:,self.bgr], cv2.CV_64F).var() #adjust bgr?
        # return cv2.boxFilter(image, ksize = (self.kernel_size, self.kernel_size), ddepth = -1)

    def apply(self, tile):
        blur = self.F(tile.image)
        if blur > self.blur_thresh:
            tile.label_name=self.blur_label
        

# Create dask cluster with Dask jupyter dashboard and then connect

Unable to get SLURMCluster working:

In [ ]:
cluster = SLURMCluster(cores=2,
                       processes=1,
                       interface="lo",  
                       memory="1g",
                       account="rbao",
                       walltime="01:00:00",
                       queue="normal")
# no workers assigned!

# cluster.scale(16)
# client = Client(cluster)

In [10]:
print(client.dashboard_link, os.getenv('SLURMD_NODENAME'))

http://10.201.8.150:8787/status htc-1024-n0


In [ ]:
client = Client(cluster)

In [40]:
cluster = LocalCluster(n_workers=4,
                        threads_per_worker=16,
                        memory_limit='32g',) #4,16,32g --> about as fast as I can get it to be with this setup. (4-8 proc at once)

/opt/conda/envs/py38/lib/python3.8/site-packages/distributed/node.py:180: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44096 instead
  warnings.warn(


In [41]:
client = Client(cluster)
print(client.dashboard_link, os.getenv('SLURMD_NODENAME'))

http://127.0.0.1:44096/status htc-1024-n0


In [32]:
client.scheduler_info()

{'type': 'Scheduler',
 'id': 'Scheduler-2a8be2b1-2a53-4afe-85cf-fd45557048d1',
 'address': 'tcp://127.0.0.1:45163',
 'services': {'dashboard': 44798},
 'started': 1695401336.492358,
 'workers': {'tcp://127.0.0.1:39162': {'type': 'Worker',
   'id': 0,
   'host': '127.0.0.1',
   'resources': {},
   'local_directory': '/scratch/slurm-2166990/dask-worker-space/dask-worker-space/worker-i5gri74l',
   'name': 0,
   'nthreads': 32,
   'memory_limit': 64000000000,
   'last_seen': 1695401364.1084597,
   'services': {'dashboard': 43095},
   'metrics': {'executing': 0,
    'in_memory': 0,
    'ready': 0,
    'in_flight': 0,
    'bandwidth': {'total': 100000000, 'workers': {}, 'types': {}},
    'spilled_nbytes': {'memory': 0, 'disk': 0},
    'cpu': 2.0,
    'memory': 80867328,
    'time': 1695401363.6318502,
    'read_bytes': 36994563.17022318,
    'write_bytes': 150226.26316886776,
    'read_bytes_disk': 0.0,
    'write_bytes_disk': 0.0,
    'num_fds': 23},
   'nanny': 'tcp://127.0.0.1:36196'},
  'tcp://127.0.0.1:46243': {'type': 'Worker',
   'id': 1,
   'host': '127.0.0.1',
   'resources': {},
   'local_directory': '/scratch/slurm-2166990/dask-worker-space/dask-worker-space/worker-i1i8621y',
   'name': 1,
   'nthreads': 32,
   'memory_limit': 64000000000,
   'last_seen': 1695401364.1284645,
   'services': {'dashboard': 46144},
   'metrics': {'executing': 0,
    'in_memory': 0,
    'ready': 0,
    'in_flight': 0,
    'bandwidth': {'total': 100000000, 'workers': {}, 'types': {}},
    'spilled_nbytes': {'memory': 0, 'disk': 0},
    'cpu': 2.0,
    'memory': 79568896,
    'time': 1695401363.6296098,
    'read_bytes': 36808085.352133095,
    'write_bytes': 150971.48683673432,
    'read_bytes_disk': 0.0,
    'write_bytes_disk': 0.0,
    'num_fds': 23},
   'nanny': 'tcp://127.0.0.1:36042'}}}

#from dashboard:
client = Client("tcp://127.0.0.1:33767") #Port depends on dashboard in this case

In [43]:
client.shutdown()

distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


# Run timing test on DASK

Notes: Adaptive workers eventually failed. Scale to 60 workers became unresponsive.

In [ ]:
a = time.time()
pipeline = Pipeline([blank_detect,art_detect,tissue_detect,normalize]) 
# try:
wsi.run(pipeline, 
        tile_size = 500, #With larger tiles I am not getting errors.
        distributed=True, 
        client=client)
# except:
#     print('Some error occurred')
# print(f"Total number of tiles extracted: {len(wsi.tiles)}")
wsi.write(results.joinpath('test.h5path'))
b = time.time()
print((b-a)/60, 'minutes')

In [10]:
b = time.time()
print((b-a)/60, 'minutes')

77.26244602998098 minutes


In [ ]:
print(f"Total number of tiles extracted: {len(wsi.tiles)}")

In [8]:
wsi.write(sp.joinpath('test.h5path'))

In [11]:
wsi.write(results.joinpath('test.h5path'))

# Load h5 as a dataloader

In [16]:
dataset = TileDataset(sp.joinpath('test.h5path'))
dataloader = DataLoader(dataset, batch_size = 350, 
                        shuffle = False, 
                        num_workers = 1)

In [17]:
batch = next(iter(dataloader))

StopIteration: 

In [ ]:
bat

Also TODO: load 